# 🏟️ Tactical GNN — Step 1: 데이터 파이프라인

> SoccerNet 트래킹 데이터 + AI Hub 전술 라벨 → GNN 학습용 그래프 데이터셋 구축

**GPU 필요**: Runtime → Change runtime type → T4 GPU

**Google Drive 마운트 필요**: 데이터 저장용

## 0. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi

# 패키지 설치
!pip install -q SoccerNet torch torch-geometric torch-scatter torch-sparse \
    pandas numpy matplotlib scikit-learn tqdm

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/tactical-gnn'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
SOCCERNET_DIR = os.path.join(DATA_DIR, 'soccernet')
AIHUB_DIR = os.path.join(DATA_DIR, 'aihub')
GRAPH_DIR = os.path.join(DATA_DIR, 'graphs')

for d in [PROJECT_DIR, DATA_DIR, SOCCERNET_DIR, AIHUB_DIR, GRAPH_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Project: {PROJECT_DIR}')

## 1. SoccerNet 트래킹 데이터 다운로드

### 사전 준비
1. https://www.soccer-net.org/ 에서 계정 생성
2. NDA 서명 완료
3. 아래 셀에서 로그인

In [ ]:
from SoccerNet.Downloader import SoccerNetDownloader

downloader = SoccerNetDownloader(LocalDirectory=SOCCERNET_DIR)

# 로그인 (NDA 서명 후 받은 이메일/비밀번호)
downloader.password = input('SoccerNet Password: ')

In [ ]:
# 트래킹 데이터 다운로드 (선수 좌표 — GNN의 핵심 입력)
downloader.downloadDataTask(
    task='tracking',
    split=['train', 'valid', 'test']
)
print('✅ 트래킹 데이터 다운로드 완료')

In [ ]:
# 액션 스포팅 라벨 다운로드 (이벤트 어노테이션)
downloader.downloadGames(
    files=['Labels-v2.json'],
    split=['train', 'valid', 'test']
)
print('✅ 액션 라벨 다운로드 완료')

## 2. AI Hub 데이터 로드

### 사전 준비
1. https://aihub.or.kr 에서 '전술 판정 영상 데이터(축구)' 신청
2. 승인 후 **좌표 CSV + JSON 라벨**만 다운로드 (영상은 용량 초과)
3. Google Drive의 `tactical-gnn/data/aihub/` 폴더에 업로드

업로드할 파일:
- `tracking_coordinates.csv` (선수 좌표)
- `tactical_labels.json` (전술 분류 라벨)
- `stat_events.json` (이벤트 라벨)

In [ ]:
import glob

# AI Hub 파일 확인
aihub_files = glob.glob(os.path.join(AIHUB_DIR, '**/*'), recursive=True)
print(f'AI Hub 파일 수: {len(aihub_files)}')
for f in aihub_files[:20]:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {os.path.basename(f)} ({size_mb:.1f} MB)')

## 3. 트래킹 데이터 → 그래프 변환

핵심: 각 프레임의 선수 좌표를 **그래프**로 변환

```
Node (선수): features = [x, y, vx, vy, team_id, role_id]
Edge (관계): features = [distance, relative_speed, same_team]
Label: 전술 분류 (포메이션/전략)
```

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data, InMemoryDataset
from scipy.spatial.distance import cdist
from tqdm import tqdm
import json


def frame_to_graph(
    positions: np.ndarray,
    team_ids: np.ndarray,
    ball_pos: np.ndarray | None = None,
    prev_positions: np.ndarray | None = None,
    k_neighbors: int = 5,
) -> Data:
    """단일 프레임의 선수 좌표를 PyG 그래프로 변환.

    Args:
        positions: (N, 2) 선수 좌표 [x, y]
        team_ids: (N,) 팀 ID (0 or 1)
        ball_pos: (2,) 공 좌표
        prev_positions: (N, 2) 이전 프레임 좌표 (속도 계산용)
        k_neighbors: k-nearest neighbor 엣지 수

    Returns:
        PyG Data 객체
    """
    n_players = len(positions)

    # --- Node features ---
    # 속도 계산
    if prev_positions is not None:
        velocities = positions - prev_positions  # (N, 2)
    else:
        velocities = np.zeros_like(positions)

    # 공과의 거리
    if ball_pos is not None:
        ball_dist = np.linalg.norm(positions - ball_pos, axis=1, keepdims=True)
    else:
        ball_dist = np.zeros((n_players, 1))

    # [x, y, vx, vy, team_id, ball_distance]
    node_features = np.hstack([
        positions,           # (N, 2)
        velocities,          # (N, 2)
        team_ids.reshape(-1, 1),  # (N, 1)
        ball_dist,           # (N, 1)
    ])  # → (N, 6)

    # --- Edge construction (k-NN) ---
    dist_matrix = cdist(positions, positions)  # (N, N)
    edges_src, edges_dst = [], []
    edge_features = []

    for i in range(n_players):
        # k nearest neighbors (자기 자신 제외)
        dists = dist_matrix[i]
        neighbors = np.argsort(dists)[1:k_neighbors + 1]

        for j in neighbors:
            edges_src.append(i)
            edges_dst.append(j)

            # Edge features: [distance, relative_speed, same_team]
            dist = dists[j]
            rel_speed = np.linalg.norm(velocities[i] - velocities[j])
            same_team = float(team_ids[i] == team_ids[j])
            edge_features.append([dist, rel_speed, same_team])

    edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
    edge_attr = torch.tensor(edge_features, dtype=torch.float)
    x = torch.tensor(node_features, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


# 테스트: 가상 데이터로 그래프 생성
test_positions = np.random.rand(22, 2) * 100  # 22명 선수
test_teams = np.array([0]*11 + [1]*11)
test_ball = np.array([50.0, 50.0])

g = frame_to_graph(test_positions, test_teams, test_ball)
print(f'Graph: {g.num_nodes} nodes, {g.num_edges} edges')
print(f'Node features shape: {g.x.shape}')
print(f'Edge features shape: {g.edge_attr.shape}')
print('✅ 그래프 변환 정상')

## 4. SoccerNet 트래킹 → 그래프 데이터셋 변환

In [ ]:
def parse_soccernet_tracking(tracking_dir: str) -> list[dict]:
    """SoccerNet 트래킹 데이터를 파싱하여 프레임별 선수 좌표를 추출.

    Returns:
        [{frame_id, positions, team_ids, ball_pos}, ...]
    """
    frames = []
    tracking_files = glob.glob(os.path.join(tracking_dir, '**/*.json'), recursive=True)

    if not tracking_files:
        # CSV 형식일 수 있음
        tracking_files = glob.glob(os.path.join(tracking_dir, '**/*.csv'), recursive=True)

    print(f'발견된 트래킹 파일: {len(tracking_files)}개')

    for filepath in tqdm(tracking_files[:10], desc='Parsing tracking'):
        ext = os.path.splitext(filepath)[1].lower()

        if ext == '.json':
            with open(filepath) as f:
                data = json.load(f)
            # SoccerNet JSON 형식에 맞게 파싱
            if isinstance(data, list):
                for entry in data:
                    if 'bbox' in entry or 'position' in entry:
                        frames.append(entry)

        elif ext == '.csv':
            df = pd.read_csv(filepath)
            # 프레임별 그룹핑
            if 'frame' in df.columns or 'frame_id' in df.columns:
                frame_col = 'frame' if 'frame' in df.columns else 'frame_id'
                for frame_id, group in df.groupby(frame_col):
                    pos_cols = [c for c in ['x', 'y', 'pos_x', 'pos_y'] if c in group.columns]
                    if len(pos_cols) >= 2:
                        frames.append({
                            'frame_id': int(frame_id),
                            'positions': group[pos_cols[:2]].values,
                            'team_ids': group['team_id'].values if 'team_id' in group.columns else np.zeros(len(group)),
                            'source': os.path.basename(filepath),
                        })

    print(f'파싱된 프레임: {len(frames)}개')
    return frames


# 실행 (데이터 다운로드 후)
if os.path.exists(SOCCERNET_DIR) and os.listdir(SOCCERNET_DIR):
    sn_frames = parse_soccernet_tracking(SOCCERNET_DIR)
else:
    print('⚠️ SoccerNet 데이터를 먼저 다운로드하세요 (Step 1)')
    sn_frames = []

In [ ]:
def build_graph_dataset(
    frames: list[dict],
    window_size: int = 10,
    stride: int = 5,
) -> list[Data]:
    """프레임 시퀀스를 Temporal Graph 시퀀스로 변환.

    Args:
        frames: 파싱된 프레임 리스트
        window_size: 시간 윈도우 크기 (프레임 수)
        stride: 윈도우 이동 간격

    Returns:
        PyG Data 리스트 (각 Data는 window_size 프레임의 시퀀스)
    """
    graphs = []

    for start in tqdm(range(0, len(frames) - window_size, stride), desc='Building graphs'):
        window = frames[start:start + window_size]

        # 윈도우 내 각 프레임을 그래프로 변환
        frame_graphs = []
        for i, frame in enumerate(window):
            positions = frame['positions']
            team_ids = frame['team_ids']
            ball_pos = frame.get('ball_pos')
            prev_pos = window[i-1]['positions'] if i > 0 else None

            g = frame_to_graph(positions, team_ids, ball_pos, prev_pos)
            frame_graphs.append(g)

        # 시퀀스를 하나의 Data로 결합
        # x: (window_size, N, F) → (window_size * N, F) with batch index
        all_x = torch.cat([g.x for g in frame_graphs], dim=0)
        all_edge_attr = torch.cat([g.edge_attr for g in frame_graphs], dim=0)

        # edge_index 오프셋 조정
        all_edges = []
        offset = 0
        temporal_batch = []
        for t, g in enumerate(frame_graphs):
            all_edges.append(g.edge_index + offset)
            temporal_batch.extend([t] * g.num_nodes)
            offset += g.num_nodes

        all_edge_index = torch.cat(all_edges, dim=1)
        temporal_batch = torch.tensor(temporal_batch, dtype=torch.long)

        combined = Data(
            x=all_x,
            edge_index=all_edge_index,
            edge_attr=all_edge_attr,
            temporal_batch=temporal_batch,
            num_frames=window_size,
        )
        graphs.append(combined)

    print(f'✅ 그래프 데이터셋: {len(graphs)}개 시퀀스')
    return graphs


# 가상 데이터로 테스트
dummy_frames = []
for i in range(100):
    dummy_frames.append({
        'frame_id': i,
        'positions': np.random.rand(22, 2) * 105,  # 축구장 105m x 68m
        'team_ids': np.array([0]*11 + [1]*11),
        'ball_pos': np.random.rand(2) * np.array([105, 68]),
    })

dummy_graphs = build_graph_dataset(dummy_frames, window_size=10, stride=5)
print(f'Sample graph: {dummy_graphs[0]}')

## 5. AI Hub 전술 라벨 통합

In [ ]:
# AI Hub 전술 분류 체계
TACTICAL_LABELS = {
    # 포메이션
    'formation_433': 0,
    'formation_442': 1,
    'formation_352': 2,
    'formation_4231': 3,
    'formation_343': 4,
    'formation_532': 5,
    'formation_4141': 6,
    # 전술 상태
    'high_press': 7,
    'counter_attack': 8,
    'possession_play': 9,
    'defensive_block': 10,
    'transition_attack': 11,
    'transition_defense': 12,
    'set_piece': 13,
}

NUM_CLASSES = len(TACTICAL_LABELS)
print(f'전술 분류 클래스 수: {NUM_CLASSES}')
for name, idx in TACTICAL_LABELS.items():
    print(f'  {idx}: {name}')

In [ ]:
def load_aihub_labels(aihub_dir: str) -> dict[str, int]:
    """AI Hub 전술 라벨을 로드.

    Returns:
        {match_id_frame_id: tactical_label_index}
    """
    labels = {}
    label_files = glob.glob(os.path.join(aihub_dir, '**/tactical_labels*.json'), recursive=True)

    if not label_files:
        print('⚠️ AI Hub 라벨 파일이 없습니다. 가상 라벨을 생성합니다.')
        # 가상 라벨 (실제 데이터 없을 때 학습 파이프라인 테스트용)
        for i in range(1000):
            labels[f'dummy_{i}'] = np.random.randint(0, NUM_CLASSES)
        return labels

    for filepath in label_files:
        with open(filepath) as f:
            data = json.load(f)
        if isinstance(data, list):
            for entry in data:
                key = f"{entry.get('match_id', 'unknown')}_{entry.get('frame_id', 0)}"
                label_name = entry.get('tactical_class', entry.get('category_1', ''))
                labels[key] = TACTICAL_LABELS.get(label_name, 0)

    print(f'✅ 로드된 라벨: {len(labels)}개')
    return labels


tactical_labels = load_aihub_labels(AIHUB_DIR)

## 6. 데이터셋 저장 (PyG InMemoryDataset)

In [ ]:
import torch

# 가상 라벨을 그래프에 할당
for i, g in enumerate(dummy_graphs):
    g.y = torch.tensor([np.random.randint(0, NUM_CLASSES)], dtype=torch.long)

# 저장
save_path = os.path.join(GRAPH_DIR, 'tactical_graphs.pt')
torch.save(dummy_graphs, save_path)
print(f'✅ 저장 완료: {save_path}')
print(f'   그래프 수: {len(dummy_graphs)}')
print(f'   샘플: nodes={dummy_graphs[0].num_nodes}, edges={dummy_graphs[0].num_edges}')
print(f'   파일 크기: {os.path.getsize(save_path) / 1024 / 1024:.1f} MB')

## 다음 단계

→ **02_gnn_model.ipynb** 에서 이 데이터로 Temporal GAT 모델을 학습합니다.